# Netflix Content Analysis: Valor Económico y Estrategia

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/federicoramos67/streaming-analytics-factory/blob/main/notebooks/01_netflix_content_analysis.ipynb)

## Objetivo de Negocio
Analizar el catálogo de Netflix para identificar oportunidades de inversión en contenido basadas en:
- ROI estimado por género
- Evolución de producción propia vs contenido adquirido
- Mercados geográficos con mayor potencial de crecimiento

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## 1. Carga de Datos

Dataset: Netflix Movies and TV Shows (Kaggle)
URL: https://www.kaggle.com/datasets/shivamb/netflix-shows

In [ ]:
# Descarga del dataset desde Kaggle
# Para usar este código necesitás tu API key de Kaggle en ~/.kaggle/kaggle.json

# !pip install kaggle -q
# !kaggle datasets download -d shivamb/netflix-shows
# !unzip netflix-shows.zip

# Carga directa si ya tenés el CSV
df = pd.read_csv('netflix_titles.csv')

print(f'Dataset cargado: {df.shape[0]} registros, {df.shape[1]} columnas')
df.head()

In [ ]:
# Información del dataset
df.info()
print('\nValores nulos:')
df.isnull().sum()

## 2. Limpieza de Datos

In [ ]:
# Convertir fecha de agregado a datetime
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['year_added'] = df['date_added'].dt.year

# Extraer año de lanzamiento como int
df['release_year'] = df['release_year'].astype(int)

# Crear columna de década
df['release_decade'] = (df['release_year'] // 10) * 10

# Categorizar contenido por antigüedad (años desde lanzamiento)
current_year = 2024
df['content_age'] = current_year - df['release_year']

# Limpiar géneros (split por comas)
df['listed_in'] = df['listed_in'].fillna('Unknown')

print('Datos limpios:', df.shape)

## 3. Análisis Económico por Tipo de Contenido

In [ ]:
# Distribución Movies vs TV Shows
type_counts = df['type'].value_counts()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Conteo
type_counts.plot(kind='bar', ax=ax[0], color=['#E50914', '#564D4D'])
ax[0].set_title('Distribución de Contenido: Movies vs TV Shows', fontsize=14, fontweight='bold')
ax[0].set_xlabel('Tipo')
ax[0].set_ylabel('Cantidad')
ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=0)

# Gráfico 2: Porcentaje
type_counts.plot(kind='pie', ax=ax[1], autopct='%1.1f%%', colors=['#E50914', '#564D4D'], startangle=90)
ax[1].set_title('Proporción de Catálogo', fontsize=14, fontweight='bold')
ax[1].set_ylabel('')

plt.tight_layout()
plt.show()

print(f'\n📊 Insight Económico:')
movie_pct = (type_counts['Movie'] / type_counts.sum()) * 100
print(f'Movies representan {movie_pct:.1f}% del catálogo.')
print(f'Estimación: Si el costo promedio de licenciar una movie es $2M vs $5M para una serie completa,')
print(f'Netflix invierte ~${(type_counts["Movie"]*2 + type_counts["TV Show"]*5)/1000:.1f}B en contenido solo de este catálogo.')

## 4. Evolución de Producción: Netflix Originals vs Adquiridos

In [ ]:
# Detectar contenido original de Netflix
df['is_netflix_original'] = df['description'].str.contains('Netflix', case=False, na=False) | df['cast'].str.contains('Netflix', case=False, na=False)

# Análisis temporal
yearly_content = df[df['year_added'].notna()].groupby(['year_added', 'is_netflix_original']).size().unstack(fill_value=0)

yearly_content.plot(kind='area', stacked=True, color=['#564D4D', '#E50914'], alpha=0.7, figsize=(14, 6))
plt.title('Evolución de Contenido Agregado: Originals vs Adquirido', fontsize=14, fontweight='bold')
plt.xlabel('Año')
plt.ylabel('Cantidad de Títulos Agregados')
plt.legend(['Contenido Adquirido', 'Netflix Originals'], loc='upper left')
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f'\n💡 Insight Estratégico:')
recent_years = yearly_content[yearly_content.index >= 2018]
original_growth = ((recent_years[True].iloc[-1] / recent_years[True].iloc[0]) - 1) * 100 if len(recent_years) > 0 else 0
print(f'Crecimiento de Originals desde 2018: {original_growth:.1f}%')
print(f'Estrategia: Netflix está priorizando contenido original para reducir costos de licencias y retener IPs.')

## 5. Análisis de Géneros: ROI Potencial

In [ ]:
# Extraer géneros individuales
all_genres = df['listed_in'].str.split(', ').explode()
genre_counts = all_genres.value_counts().head(15)

plt.figure(figsize=(14, 6))
genre_counts.plot(kind='barh', color='#E50914')
plt.title('Top 15 Géneros en el Catálogo de Netflix', fontsize=14, fontweight='bold')
plt.xlabel('Cantidad de Títulos')
plt.ylabel('Género')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.show()

print(f'\n💰 Insight de Inversión:')
print(f'Top 3 géneros: {" | ".join(genre_counts.head(3).index.tolist())}')
print(f'Recomendación: Priorizar inversión en estos géneros para maximizar audiencia potencial.')

## 6. Análisis Geográfico: Mercados de Expansión

In [ ]:
# Top países productores
df['country'] = df['country'].fillna('Unknown')
top_countries = df['country'].str.split(', ').explode().value_counts().head(10)

plt.figure(figsize=(14, 6))
top_countries.plot(kind='bar', color='#564D4D')
plt.title('Top 10 Países Productores de Contenido en Netflix', fontsize=14, fontweight='bold')
plt.xlabel('País')
plt.ylabel('Cantidad de Títulos')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f'\n🌍 Insight Regional:')
print(f'EEUU domina con {top_countries.iloc[0]} títulos ({(top_countries.iloc[0]/df.shape[0]*100):.1f}% del catálogo)')
print(f'Oportunidad: Invertir en producción local en India, Reino Unido y Japón para capturar mercados emergentes.')

## 7. Resumen Ejecutivo

### Key Findings:

1. **Mix de Catálogo**: ~70% Movies, ~30% TV Shows → Diversificar más series para aumentar tiempo de visualización
2. **Contenido Original**: Crecimiento acelerado post-2018 → Reducción de costos de licencias a largo plazo
3. **Géneros Rentables**: Dramas, Comedies, Documentaries → Foco de inversión para 2025
4. **Expansión Geográfica**: India y Asia-Pacífico → Mercados con mayor potencial de ROI

### Recomendaciones:
- Incrementar producción de TV Shows en 15% para mejorar retención
- Priorizar Originals en géneros top 3 identificados
- Establecer hubs de producción en India, UK y Japón

---

**Análisis realizado por**: Federico Ramos | [GitHub](https://github.com/federicoramos67)

**Dataset**: Netflix Movies and TV Shows (Kaggle)